In [0]:
%run /Workspace/weather_notebook/nb_utils_dev

In [0]:
# ── Cell 2: Read Silver ───────────────────────────────────────
print("city_hourly_summary - layer: GOLD")
print("\n[1/3] Reading Silver tables...")

df_weather = spark.table(f"{silver_catalog}.weather_readings")
df_airquality = spark.table(f"{silver_catalog}.air_quality_readings")

print(f"  Weather rows:     {df_weather.count():,}")
print(f"  Air quality rows: {df_airquality.count():,}")

In [0]:
# ── Cell 3: Build Gold city summary ──────────────────────────
print("\n[2/3] Building Gold city summary...")

gold_city_summary = (
    df_weather
    .groupBy(
        "city_name", "latitude", "longitude",
        "reading_date", "reading_hour"
    )
    .agg(
        F.round(F.avg("temperature_c"), 2).alias("avg_temp_c"),
        F.round(F.max("temperature_c"), 2).alias("max_temp_c"),
        F.round(F.min("temperature_c"), 2).alias("min_temp_c"),
        F.round(F.avg("humidity_pct"), 2).alias("avg_humidity"),
        F.round(F.avg("pressure_hpa"), 2).alias("avg_pressure"),
        F.round(F.avg("wind_speed_ms"), 2).alias("avg_wind_speed"),
        F.round(F.max("wind_gust_ms"), 2).alias("max_wind_gust"),
        F.round(F.avg("cloud_cover_pct"), 2).alias("avg_cloud_cover"),
        F.first("weather_main").alias("weather_condition"),
        F.first("weather_severity").alias("severity"),
        F.first("wind_beaufort").alias("beaufort_scale"),
        F.first("comfort_level").alias("comfort_level"),
        F.first("is_daytime").alias("is_daytime"),
        F.first("daylight_hours").alias("daylight_hours"),
        F.count("*").alias("reading_count")
    )
    .withColumn("ingestion_date", F.current_date())
    .withColumn("ingestion_ts",   F.current_timestamp())
)

row_count = gold_city_summary.count()
print(f"  Gold rows: {row_count:,}")

In [0]:
# ── Cell 4: Write to ADLS Gen2 + register in Unity Catalog ───
print("\n[3/3] Writing to ADLS Gen2...")
row_count = write_gold_table(gold_city_summary, "city_hourly_summary")

In [0]:
# ── Cell 4: Write to ADLS Gen2 + register in Unity Catalog ───
print("\n[3/3] Writing to ADLS Gen2...")
row_count = write_gold_table(gold_city_summary, "city_hourly_summary")


In [0]:
# ── Cell 5: Verify ────────────────────────────────────────────
spark.sql(f"""
    SELECT city_name, reading_date, reading_hour,
           avg_temp_c, weather_condition,
           avg_humidity, avg_wind_speed,
           comfort_level, severity
    FROM {gold_catalog}.city_hourly_summary
    ORDER BY city_name
""").show(truncate=False)

log_footer("city_hourly_summary", row_count, "PASS", layer="GOLD")
dbutils.notebook.exit(f"city_hourly_summary|{row_count}|PASS")